# SC4001 Phase 3, equal-compute lambda sweep at the settings where the adversary can train

Accelerator must be **GPU (T4 x2 or P100)**. `src/train_dann.py` refuses CPU by design.
Code and data come from the attached dataset `isaacwjk21/sc4001-project`; no internet or installs.

## Why this kernel exists

Three Phase 3 kernels have run and none has measured DANN.

1. `sc4001-phase3-davis-dann` applied lambda twice, once as the gradient reversal coefficient
   and again as a weight on the domain BCE, so the discriminator trained at about 2.5% of its
   nominal rate and never became an adversary. Result invalid, archived under
   `outputs/phase3-lambda-squared/`.
2. `sc4001-phase3-davis-lambda` swept lambda over a tenfold range under the corrected loss and
   got the same test MSE every time, 0.4949 / 0.4946 / 0.4930 against the baseline's 0.383.
   Lambda being irrelevant was the tell: early stopping at `patience` 15 had truncated every
   DANN run at epoch 18 to 21 while every baseline ran to 66 to 100.
3. `sc4001-phase3-davis-nostop` removed the stopping rule, ran both arms for a fixed 100
   epochs, and showed the deficit was training length: paired by seed, DANN was +0.0112 test
   MSE (t(2) = +1.82) and +0.0038 CI (t(2) = +0.96), neither significant at three seeds. But it
   ran at `lambda_max` 1.0, where `domain_loss` stayed at 0.691 to 0.692 against
   ln 2 = 0.693147 and `domain_accuracy` at 0.51 to 0.53 for all 100 epochs. The reversal
   suppresses the discriminator from the first step, so that arm was the baseline plus noise.

So the stopping-rule confound is gone, and what remains is that the method has only ever been
run with a dead adversary. This kernel runs it where the adversary can train.

## Design

DAVIS `split_cold_target`, seeds 4001-4003, `--patience 100` so every run gets the full 100
epochs and the best epoch is chosen from `history.csv` afterwards.
`--lambda-max` in {0.03, 0.1, 0.3}, nine runs. Each lambda gets its own `SC4001_OUTPUT_ROOT`
under `outputs/lambda-nostop/lambda<value>/`, because the per-run output path carries no lambda.

Lambda 0.1 is included because it is the one setting at which the discriminator demonstrably
learned: in the truncated sweep its `domain_loss` fell from 0.692 to 0.647 and accuracy reached
0.62. Note that this is not a repeat of that run. Ganin's schedule reaches only 0.077 of
`lambda_max` by epoch 20 but essentially all of it by epoch 100, so over a full budget lambda
0.1 applies far more adversarial pressure late in training than the truncated run ever did.
0.03 brackets it from below in case 0.1 also ends up suppressing the discriminator once the
schedule saturates, and 0.3 brackets from above to find where suppression starts.

**The baseline arm is not re-run.** `sc4001-phase3-davis-nostop` already produced a
patience-100 baseline on the same split and the same three seeds with the same epoch budget:
test MSE 0.3991 (sd 0.0217), test CI 0.7885 (sd 0.0014), best validation MSE 0.4621. That is
the comparison arm, and its per-seed values are printed at the end of this notebook for pairing.
Re-running it would cost an hour of quota to reproduce numbers already in hand.

## What the output answers

The grid cell prints a **validity gate before any MSE**: per run, `domain_loss` at epochs 1, 10,
25, 50, 75 and 100 against ln 2, the minimum loss and maximum accuracy over the whole run, and
a verdict of TRAINED or AT CHANCE. A run is counted as having trained only if `domain_loss`
falls at least 0.005 below ln 2 at some point or accuracy exceeds 0.55.

If no run trains, the MSE table that follows measures the baseline plus unlabelled forward
passes, the notebook says so explicitly, and the conclusion is about the architecture or the
schedule rather than about domain adaptation. If some lambda does train, then its test MSE and
CI against the stored baseline are the project's first valid statement about whether adversarial
domain alignment helps cold-target drug-target affinity regression, in either direction.

Expect roughly three hours: 11.83 s per epoch measured for DANN on this split, so about 20 min
per run across nine runs.

Outputs survive only if this version is saved.


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise SystemExit("No CUDA device. Set the accelerator to GPU T4 x2 or P100, not TPU or None.")
props = torch.cuda.get_device_properties(0)
print("torch", torch.__version__, "| device", props.name,
      "|", round(props.total_memory / 1e9, 1), "GB | count", torch.cuda.device_count())


In [ ]:
# Show what is actually mounted, so a layout surprise is visible rather than guessed at.
inputs = sorted(Path("/kaggle/input").rglob("*"))
print(len(inputs), "mounted paths; first 25:")
for path in inputs[:25]:
    print("   ", path)

# Locate the project root: the directory holding both src/train.py and data/processed/vocab.json.
# Kaggle strips the archive's top-level folder, so do not assume a "Project" path segment.
candidates = [p.parents[1] for p in Path("/kaggle/input").rglob("src/train.py")]
roots = [p for p in candidates if (p / "data" / "processed" / "vocab.json").exists()]
if not roots:
    raise SystemExit(
        "No project root found. Looked for src/train.py beside data/processed/vocab.json. "
        "Candidates with src/train.py but no vocab.json: " + str(candidates)
    )
source = sorted(roots)[0]

project = Path("/kaggle/working/sc4001/Project")
shutil.copytree(source, project, dirs_exist_ok=True)
env = dict(os.environ, SC4001_OUTPUT_ROOT="/kaggle/working/outputs", PYTHONUNBUFFERED="1")
print("source :", source)
print("project:", project)
print("tables :", sorted(p.name for p in (project / "data" / "processed").iterdir()))

# Phase 3 needs the external kinase pool and the extra split columns, not just the Phase 2 tables.
required = ["external_kinases.csv", "external_tokens_davis.npz", "external_tokens_kiba.npz",
            "davis_extra_splits.csv", "kiba_extra_splits.csv"]
present = {p.name for p in (project / "data" / "processed").iterdir()}
missing = [name for name in required if name not in present and name + ".gz" not in present]
if missing:
    raise SystemExit("Dataset is missing Phase 3 inputs: " + str(missing))
print("phase 3 inputs OK")


In [ ]:
# Smoke the entry point on the split being run before spending quota, so a loader or
# split-resolution fault costs seconds rather than an hour.
smoke = subprocess.run(
    [sys.executable, "src/train_dann.py", "--dataset", "davis",
     "--split", "split_cold_target", "--smoke"],
    cwd=project, env=env, check=False,
)
if smoke.returncode != 0:
    raise SystemExit("Smoke run of src/train_dann.py failed with code " + str(smoke.returncode)
                     + "; not starting the grid.")
for stale in Path("/kaggle/working/outputs").rglob("*-smoke"):
    shutil.rmtree(stale, ignore_errors=True)
print("smoke OK")


In [ ]:
# Equal-compute lambda sweep. Every run gets the full 100 epochs (--patience 100) so the
# stopping rule cannot decide the comparison, and the baseline arm is NOT re-run: the
# patience-100 baseline from sc4001-phase3-davis-nostop (same split, same seeds, same epoch
# budget) is the comparison arm and lives at outputs/nostop/baseline/.
# The output path carries no lambda, so each lambda needs its own SC4001_OUTPUT_ROOT.
ROOT = "/kaggle/working/outputs/lambda-nostop"
LAMBDAS = ("0.03", "0.1", "0.3")
GRID = [(lam, seed) for lam in LAMBDAS for seed in (4001, 4002, 4003)]

started = time.time()
status = []
for lam, seed in GRID:
    t0 = time.time()
    label = "lambda" + lam + " davis split_cold_target seed" + str(seed)
    print()
    print("===== " + label + " (patience 100, no early stop) =====", flush=True)
    run = subprocess.run(
        [sys.executable, "src/train_dann.py", "--dataset", "davis",
         "--split", "split_cold_target", "--seed", str(seed),
         "--patience", "100", "--lambda-max", lam,
         "--discriminator-input", "protein", "--resume"],
        cwd=project, env=dict(env, SC4001_OUTPUT_ROOT=ROOT + "/lambda" + lam), check=False,
    )
    elapsed = time.time() - t0
    status.append((label, run.returncode, elapsed))
    print("----- rc=" + str(run.returncode) + " wall=" + format(elapsed / 60, ".1f")
          + " min -----", flush=True)

print()
print("Grid wall time", format((time.time() - started) / 60, ".1f"), "min")
for label, rc, elapsed in status:
    print("%-52s rc=%d %6.1f min" % (label, rc, elapsed / 60))
failed = [row for row in status if row[1] != 0]
print("FAILED RUNS:", failed if failed else "none")

# THE VALIDITY GATE. A DANN number means nothing if the discriminator never trained. At
# lambda_max 1.0 over 100 epochs, domain_loss sat at 0.691-0.692 against ln 2 = 0.693131 and
# accuracy at 0.51-0.53, so that arm was the baseline plus noise. Print the trajectory and say
# per run whether the adversary moved, before any MSE is read.
LN2 = 0.6931471805599453
print()
print("ADVERSARY CHECK: domain_loss against ln 2 = %.6f" % LN2)
print("%-10s %5s %9s %9s %9s %9s %9s %9s  %s"
      % ("lambda", "seed", "ep1", "ep10", "ep25", "ep50", "ep75", "ep100", "verdict"))
gate = []
for path in sorted(Path(ROOT).glob("**/history.csv")):
    rows = [line.split(",") for line in path.read_text().strip().splitlines()]
    header, body = rows[0], rows[1:]
    if not body:
        continue
    index = {name: position for position, name in enumerate(header)}
    by_epoch = {int(row[index["epoch"]]): row for row in body}
    picks = [by_epoch.get(e) for e in (1, 10, 25, 50, 75, 100)]
    losses = [None if row is None else float(row[index["domain_loss"]]) for row in picks]
    accuracies = [float(row[index["domain_accuracy"]]) for row in body]
    # Departed from chance if domain_loss ever falls meaningfully below ln 2, or accuracy
    # ever rises well above 0.5. Both directions matter: a loss above ln 2 with accuracy at
    # chance still means a suppressed adversary, not a defeated one.
    low = min(float(row[index["domain_loss"]]) for row in body)
    high = max(accuracies)
    trained = (LN2 - low) > 0.005 or high > 0.55
    lam = str(path).replace(ROOT + "/", "").split("/")[0]
    seed = int(str(path.parent).split("seed")[-1])
    gate.append((lam, seed, trained, low, high))
    print("%-10s %5d %s  %s"
          % (lam, seed,
             " ".join("      n/a" if v is None else "%9.6f" % v for v in losses),
             ("TRAINED (min loss %.4f, max acc %.3f)" if trained
              else "AT CHANCE (min loss %.4f, max acc %.3f)") % (low, high)))

trained_any = [row for row in gate if row[2]]
print()
print("%d of %d runs had an adversary that departed from chance." % (len(trained_any), len(gate)))
if not trained_any:
    print("NO RUN HAD A WORKING ADVERSARY. The MSE numbers below do not measure DANN; they "
          "measure the baseline plus unlabelled forward passes. Do not report them as a "
          "result about domain adaptation.")

print()
print("Best validation MSE over 100 epochs:")
for path in sorted(Path(ROOT).glob("**/history.csv")):
    rows = [line.split(",") for line in path.read_text().strip().splitlines()]
    header, body = rows[0], rows[1:]
    if not body:
        continue
    index = {name: position for position, name in enumerate(header)}
    best = min(body, key=lambda row: float(row[index["val_mse"]]))
    lam = str(path).replace(ROOT + "/", "").split("/")[0]
    print("  %-10s %-22s best val_mse %.6f at epoch %-4s (final epoch %s, final val %.6f)"
          % (lam, str(path.parent).split("/")[-1], float(best[index["val_mse"]]),
             best[index["epoch"]], body[-1][index["epoch"]],
             float(body[-1][index["val_mse"]])))

print()
print("Test metrics at each run's own best checkpoint.")
print("Comparison arm: patience-100 baseline from sc4001-phase3-davis-nostop,")
print("  test MSE 0.3991 (sd 0.0217), test CI 0.7885 (sd 0.0014), best val MSE 0.4621.")
print("  Per seed: 4001 MSE 0.3991, 4002 MSE 0.4208, 4003 MSE 0.3775.")
for path in sorted(Path(ROOT).glob("**/metrics.json")):
    payload = json.loads(path.read_text())
    lam = str(path).replace(ROOT + "/", "").split("/")[0]
    print("  %-10s seed%d  test MSE %.4f  CI %.4f  val MSE %.4f  best_ep %d  lambda %s"
          % (lam, payload["seed"], payload["test"]["all"]["mse"], payload["test"]["all"]["ci"],
             payload["val"]["all"]["mse"], payload["best_epoch"],
             payload.get("lambda_max", "n/a")))


In [ ]:
# Print every metrics.json so the numbers live in the saved notebook even if files are lost.
found = sorted(Path("/kaggle/working/outputs").glob("**/metrics.json"))
print(len(found), "metrics files")
rows = []
for path in found:
    payload = json.loads(path.read_text())
    print("=" * 70)
    print(path)
    print(json.dumps(payload, indent=2))
    model = payload["test"]["all"]
    phase = "phase3" if "phase3" in str(path) else "phase2"
    rows.append((phase, payload["split"], payload["seed"], payload.get("lambda_max"),
                 model["mse"], model["ci"], payload["best_epoch"], payload["total_seconds"]))

print("=" * 100)
print("%-7s %-26s %5s %7s %9s %8s %8s %9s"
      % ("phase", "split", "seed", "lambda", "mse", "ci", "best_ep", "minutes"))
for phase, split, seed, lam, mse, ci, best_ep, secs in sorted(rows):
    lam_text = "    n/a" if lam is None else "%7.2f" % lam
    print("%-7s %-26s %5d %s %9.4f %8.4f %8d %9.1f"
          % (phase, split, seed, lam_text, mse, ci, best_ep, secs / 60))
